In [1]:
import pandas as pd
import numpy as np
import re, os, json, joblib, warnings, time
from collections import Counter

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    VotingClassifier
)
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, r2_score,
    mean_squared_error, mean_absolute_error, silhouette_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.semi_supervised import LabelPropagation
import xgboost as xgb

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── PATHS ──
CSV_DIR  = '/kaggle/input/datasets/humnasadia/quiz-dataset'
NPZ_DIR  = '/kaggle/input/datasets/humnasadia/npzz-dataset'
SAVE_DIR = '/kaggle/working/models/model_b'
os.makedirs(SAVE_DIR, exist_ok=True)

print("Setup complete!")

Setup complete!


In [2]:
start_time = time.time()

train_df = pd.read_csv(os.path.join(CSV_DIR, 'train_clean.csv'))
dev_df   = pd.read_csv(os.path.join(CSV_DIR, 'dev_clean.csv'))
test_df  = pd.read_csv(os.path.join(CSV_DIR, 'test_clean.csv'))

for dset in [train_df, dev_df, test_df]:
    for col in ['A', 'B', 'C', 'D', 'A_clean', 'B_clean', 'C_clean', 'D_clean']:
        dset[col] = dset[col].fillna('')

print(f"Train: {train_df.shape}")
print(f"Dev:   {dev_df.shape}")
print(f"Test:  {test_df.shape}")
print(f"Train answer dist: {train_df['answer'].value_counts().to_dict()}")

STOPWORDS = frozenset('the a an is are was were in on at to for of and or but it he she they we i you that this with from by as not be has have had do does did will would can could may might shall should about up out if then than what which who whom when where why how all each every both few more most other some such no nor only own same so very just because its my his her their our your also been into'.split())

def clean_text(text):
    if not isinstance(text, str): return ""
    return re.sub(r'[^\w\s]', '', text.lower()).strip()

def tokenize(text):
    return clean_text(text).split()

def content_tokens(text):
    return set(tokenize(text)) - STOPWORDS

def jaccard(a, b):
    sa, sb = set(tokenize(a)), set(tokenize(b))
    u = sa | sb
    return len(sa & sb) / len(u) if u else 0.0

def overlap_ratio(a, b):
    ta, tb = set(tokenize(a)), set(tokenize(b))
    return len(ta & tb) / len(ta) if ta else 0.0

def sentence_split(article):
    sents = re.split(r'(?<=[.!?])\s+', str(article))
    return [s.strip() for s in sents if len(s.strip()) > 10]

print("Data loaded & helpers ready!")

Train: (70270, 54)
Dev:   (8786, 54)
Test:  (8786, 54)
Train answer dist: {'C': 19082, 'B': 18134, 'D': 17800, 'A': 15254}
Data loaded & helpers ready!


In [3]:
print("=" * 70)
print("BUILDING DISTRACTOR RANKING FEATURES")
print("=" * 70)

def build_distractor_features(df, desc=""):
    rows = []
    for _, r in df.iterrows():
        article      = str(r['article'])
        question     = str(r['question'])
        ans_label    = r['answer']
        correct_text = str(r[ans_label])
        sents = sentence_split(article)
        correct_ct = content_tokens(correct_text)
        q_ct       = content_tokens(question)

        for opt in 'ABCD':
            opt_text  = str(r[opt])
            opt_ct    = content_tokens(opt_text)
            is_dist   = int(opt != ans_label)

            ohe_cos   = r.get(f'ohe_cos_art_{opt}', 0.0)
            tfidf_cos = r.get(f'tfidf_cos_art_{opt}', 0.0)
            jac_q     = r.get(f'jaccard_q_{opt}', 0.0)
            art_ov    = r.get(f'{opt}_article_overlap', 0.0)
            q_ov      = r.get(f'{opt}_question_overlap', 0.0)
            wc        = r.get(f'{opt}_word_count', len(tokenize(opt_text)))

            correct_wc   = len(tokenize(correct_text))
            len_ratio    = wc / max(correct_wc, 1)
            char_count   = len(opt_text)
            correct_cc   = len(correct_text)
            cc_ratio     = char_count / max(correct_cc, 1)
            correct_jac  = jaccard(opt_text, correct_text)
            correct_ov   = overlap_ratio(opt_text, correct_text)
            shared_content  = len(opt_ct & correct_ct) / max(len(opt_ct | correct_ct), 1)
            q_content_share = len(opt_ct & q_ct) / max(len(opt_ct | q_ct), 1)

            opt_clean = clean_text(opt_text)
            in_article = 1 if opt_clean in clean_text(article) else 0
            in_sentence = 1 if any(opt_clean in clean_text(s) for s in sents) else 0

            pos = article.lower().find(opt_clean[:min(20, len(opt_clean))])
            position = pos / max(len(article), 1) if pos >= 0 else -1.0

            corr_art_ov = r.get('correct_option_art_overlap', 0.0)
            art_ov_diff = abs(float(art_ov) - float(corr_art_ov))
            unique_to_opt = len(opt_ct - q_ct - correct_ct)

            rows.append({
                'is_distractor':    is_dist,
                'ohe_cos_art':      float(ohe_cos) if pd.notna(ohe_cos) else 0.0,
                'tfidf_cos_art':    float(tfidf_cos) if pd.notna(tfidf_cos) else 0.0,
                'jaccard_q':        float(jac_q) if pd.notna(jac_q) else 0.0,
                'art_overlap':      float(art_ov),
                'q_overlap':        float(q_ov),
                'word_count':       int(wc),
                'len_ratio':        len_ratio,
                'char_count':       char_count,
                'cc_ratio':         cc_ratio,
                'correct_jaccard':  correct_jac,
                'correct_overlap':  correct_ov,
                'shared_content':   shared_content,
                'q_content_share':  q_content_share,
                'in_article':       in_article,
                'in_sentence':      in_sentence,
                'position':         position,
                'art_ov_diff':      art_ov_diff,
                'unique_to_opt':    unique_to_opt,
            })

    result_df = pd.DataFrame(rows)
    feat_cols = [c for c in result_df.columns if c != 'is_distractor']
    X = np.nan_to_num(result_df[feat_cols].values.astype(float), nan=0.0)
    y = result_df['is_distractor'].values
    if desc:
        print(f"  {desc}: {X.shape[0]} samples, {X.shape[1]} features, "
              f"correct={sum(y==0)}, distractor={sum(y==1)}")
    return X, y, feat_cols

X_train, y_train, feat_cols = build_distractor_features(train_df, "Train")
X_dev,   y_dev,   _         = build_distractor_features(dev_df,   "Dev")
X_test,  y_test,  _         = build_distractor_features(test_df,  "Test")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_dev_sc   = scaler.transform(X_dev)
X_test_sc  = scaler.transform(X_test)

print("Feature extraction complete!")

BUILDING DISTRACTOR RANKING FEATURES
  Train: 281080 samples, 18 features, correct=70270, distractor=210810
  Dev: 35144 samples, 18 features, correct=8786, distractor=26358
  Test: 35144 samples, 18 features, correct=8786, distractor=26358
Feature extraction complete!


In [4]:
print("=" * 70)
print("SUPERVISED DISTRACTOR RANKER MODELS")
print("=" * 70)

results_b1 = {}
predictions_dev = {}
predictions_test = {}

def train_eval(name, model, X_tr, y_tr, X_val, y_val, X_te, y_te):
    model.fit(X_tr, y_tr)
    p_val = model.predict(X_val)
    p_te  = model.predict(X_te)

    results_b1[name] = {
        'dev_accuracy':  accuracy_score(y_val, p_val),
        'dev_f1':        f1_score(y_val, p_val, average='macro'),
        'dev_precision': precision_score(y_val, p_val, average='macro'),
        'dev_recall':    recall_score(y_val, p_val, average='macro'),
        'test_accuracy': accuracy_score(y_te, p_te),
        'test_f1':       f1_score(y_te, p_te, average='macro'),
        'test_precision': precision_score(y_te, p_te, average='macro'),
        'test_recall':   recall_score(y_te, p_te, average='macro'),
    }
    predictions_dev[name] = p_val
    predictions_test[name] = p_te

    print(f"\n{name}")
    print(f"  Dev  — Acc: {results_b1[name]['dev_accuracy']:.4f}  F1: {results_b1[name]['dev_f1']:.4f}")
    print(f"  Test — Acc: {results_b1[name]['test_accuracy']:.4f}  F1: {results_b1[name]['test_f1']:.4f}")
    print(classification_report(y_val, p_val, target_names=['Correct', 'Distractor']))
    return model

# Logistic Regression
lr = train_eval('Logistic Regression',
    LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=42),
    X_train_sc, y_train, X_dev_sc, y_dev, X_test_sc, y_test)

# SVM
svm = train_eval('SVM (LinearSVC)',
    LinearSVC(max_iter=3000, C=1.0, class_weight='balanced', random_state=42),
    X_train_sc, y_train, X_dev_sc, y_dev, X_test_sc, y_test)

# Random Forest
rf = train_eval('Random Forest',
    RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced',
                           random_state=42, n_jobs=-1),
    X_train, y_train, X_dev, y_dev, X_test, y_test)

# XGBoost
xgb_m = train_eval('XGBoost',
    xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                       random_state=42, eval_metric='logloss', n_jobs=-1),
    X_train, y_train, X_dev, y_dev, X_test, y_test)

# Naive Bayes
nb = train_eval('Naive Bayes',
    GaussianNB(),
    X_train_sc, y_train, X_dev_sc, y_dev, X_test_sc, y_test)

SUPERVISED DISTRACTOR RANKER MODELS

Logistic Regression
  Dev  — Acc: 0.9991  F1: 0.9987
  Test — Acc: 0.9991  F1: 0.9989
              precision    recall  f1-score   support

     Correct       1.00      1.00      1.00      8786
  Distractor       1.00      1.00      1.00     26358

    accuracy                           1.00     35144
   macro avg       1.00      1.00      1.00     35144
weighted avg       1.00      1.00      1.00     35144


SVM (LinearSVC)
  Dev  — Acc: 0.9995  F1: 0.9994
  Test — Acc: 0.9994  F1: 0.9992
              precision    recall  f1-score   support

     Correct       1.00      1.00      1.00      8786
  Distractor       1.00      1.00      1.00     26358

    accuracy                           1.00     35144
   macro avg       1.00      1.00      1.00     35144
weighted avg       1.00      1.00      1.00     35144


Random Forest
  Dev  — Acc: 0.9997  F1: 0.9995
  Test — Acc: 0.9995  F1: 0.9994
              precision    recall  f1-score   support

    

In [5]:
print("=" * 70)
print("ENSEMBLE — SOFT VOTING")
print("=" * 70)

ens = VotingClassifier(estimators=[
    ('lr', LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced',
                                   random_state=42, n_jobs=-1)),
    ('xgb', xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                random_state=42, eval_metric='logloss', n_jobs=-1)),
], voting='soft')

ens = train_eval('Ensemble (Soft Vote)', ens,
    X_train_sc, y_train, X_dev_sc, y_dev, X_test_sc, y_test)

ENSEMBLE — SOFT VOTING

Ensemble (Soft Vote)
  Dev  — Acc: 0.9997  F1: 0.9996
  Test — Acc: 0.9995  F1: 0.9994
              precision    recall  f1-score   support

     Correct       1.00      1.00      1.00      8786
  Distractor       1.00      1.00      1.00     26358

    accuracy                           1.00     35144
   macro avg       1.00      1.00      1.00     35144
weighted avg       1.00      1.00      1.00     35144



In [6]:
print("=" * 70)
print("HYPERPARAMETER TUNING (XGBoost)")
print("=" * 70)

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [200, 300, 400],
}
gs = GridSearchCV(
    xgb.XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1),
    param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1
)
gs.fit(X_train, y_train)

best_xgb = gs.best_estimator_
p_dev  = best_xgb.predict(X_dev)
p_test = best_xgb.predict(X_test)
predictions_dev['XGBoost (Tuned)'] = p_dev
predictions_test['XGBoost (Tuned)'] = p_test
results_b1['XGBoost (Tuned)'] = {
    'dev_accuracy':  accuracy_score(y_dev, p_dev),
    'dev_f1':        f1_score(y_dev, p_dev, average='macro'),
    'dev_precision': precision_score(y_dev, p_dev, average='macro'),
    'dev_recall':    recall_score(y_dev, p_dev, average='macro'),
    'test_accuracy': accuracy_score(y_test, p_test),
    'test_f1':       f1_score(y_test, p_test, average='macro'),
    'test_precision': precision_score(y_test, p_test, average='macro'),
    'test_recall':   recall_score(y_test, p_test, average='macro'),
}

print(f"\nBest params: {gs.best_params_}")
print(f"Best CV F1: {gs.best_score_:.4f}")
print(f"Dev  — Acc: {results_b1['XGBoost (Tuned)']['dev_accuracy']:.4f}  F1: {results_b1['XGBoost (Tuned)']['dev_f1']:.4f}")
print(f"Test — Acc: {results_b1['XGBoost (Tuned)']['test_accuracy']:.4f}  F1: {results_b1['XGBoost (Tuned)']['test_f1']:.4f}")
print(classification_report(y_dev, p_dev, target_names=['Correct', 'Distractor']))

HYPERPARAMETER TUNING (XGBoost)
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Best params: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200}
Best CV F1: 0.9990
Dev  — Acc: 0.9997  F1: 0.9996
Test — Acc: 0.9995  F1: 0.9994
              precision    recall  f1-score   support

     Correct       1.00      1.00      1.00      8786
  Distractor       1.00      1.00      1.00     26358

    accuracy                           1.00     35144
   macro avg       1.00      1.00      1.00     35144
weighted avg       1.00      1.00      1.00     35144



In [7]:
print("=" * 70)
print("UNSUPERVISED — K-MEANS CLUSTERING")
print("=" * 70)

def cluster_purity(true, pred):
    cm = confusion_matrix(true, pred)
    return np.sum(np.max(cm, axis=0)) / np.sum(cm)

rng = np.random.RandomState(42)
sample_size_km = min(50000, len(X_train_sc))
km_idx = rng.choice(len(X_train_sc), sample_size_km, replace=False)

km = KMeans(n_clusters=2, random_state=42, n_init=10)
km.fit(X_train_sc[km_idx])

km_train_labels = km.predict(X_train_sc[km_idx])
km_dev_labels   = km.predict(X_dev_sc)
km_test_labels  = km.predict(X_test_sc)

sil_train = silhouette_score(X_train_sc[km_idx], km_train_labels, sample_size=5000)
sil_dev   = silhouette_score(X_dev_sc, km_dev_labels, sample_size=5000)

train_purity = cluster_purity(y_train[km_idx], km_train_labels)
dev_purity   = cluster_purity(y_dev, km_dev_labels)
test_purity  = cluster_purity(y_test, km_test_labels)

print(f"  Train Silhouette:  {sil_train:.4f}")
print(f"  Dev   Silhouette:  {sil_dev:.4f}")
print(f"  Train Purity:      {train_purity:.4f}")
print(f"  Dev   Purity:      {dev_purity:.4f}")
print(f"  Test  Purity:      {test_purity:.4f}")

UNSUPERVISED — K-MEANS CLUSTERING
  Train Silhouette:  0.2435
  Dev   Silhouette:  0.2320
  Train Purity:      0.9733
  Dev   Purity:      0.9753
  Test  Purity:      0.9744


In [8]:
print("=" * 70)
print("SEMI-SUPERVISED — LABEL PROPAGATION")
print("=" * 70)

sample_size_lp = min(5000, len(X_train_sc))
lp_idx = rng.choice(len(X_train_sc), sample_size_lp, replace=False)
X_lp = X_train_sc[lp_idx]
y_lp_true = y_train[lp_idx]

mask = rng.rand(sample_size_lp) > 0.2
y_lp = y_lp_true.copy()
y_lp[mask] = -1

print(f"Samples: {sample_size_lp}, Labeled: {(~mask).sum()}, Unlabeled: {mask.sum()}")

lp = LabelPropagation(kernel='knn', n_neighbors=7, max_iter=1000)
lp.fit(X_lp, y_lp)

lp_pred_unlabeled = lp.predict(X_lp[mask])
lp_acc_unlabeled  = accuracy_score(y_lp_true[mask], lp_pred_unlabeled)
lp_f1_unlabeled   = f1_score(y_lp_true[mask], lp_pred_unlabeled, average='macro')

lp_pred_dev = lp.predict(X_dev_sc)
lp_acc_dev  = accuracy_score(y_dev, lp_pred_dev)
lp_f1_dev   = f1_score(y_dev, lp_pred_dev, average='macro')

lp_pred_test = lp.predict(X_test_sc)
lp_acc_test  = accuracy_score(y_test, lp_pred_test)
lp_f1_test   = f1_score(y_test, lp_pred_test, average='macro')

print(f"On unlabeled train — Acc: {lp_acc_unlabeled:.4f}  F1: {lp_f1_unlabeled:.4f}")
print(f"On dev set         — Acc: {lp_acc_dev:.4f}  F1: {lp_f1_dev:.4f}")
print(f"On test set        — Acc: {lp_acc_test:.4f}  F1: {lp_f1_test:.4f}")

SEMI-SUPERVISED — LABEL PROPAGATION
Samples: 5000, Labeled: 1029, Unlabeled: 3971
On unlabeled train — Acc: 0.9902  F1: 0.9868
On dev set         — Acc: 0.9915  F1: 0.9888
On test set        — Acc: 0.9919  F1: 0.9894


In [9]:
print("=" * 70)
print("CONFUSION MATRICES (Dev Set)")
print("=" * 70)

for name, pred in predictions_dev.items():
    cm = confusion_matrix(y_dev, pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"{name:25s}  TN={tn:5d}  FP={fp:3d}  FN={fn:3d}  TP={tp:5d}")

print(f"\n--- Test Set ---")
for name, pred in predictions_test.items():
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"{name:25s}  TN={tn:5d}  FP={fp:3d}  FN={fn:3d}  TP={tp:5d}")

print(f"\n{'=' * 70}")
print("FEATURE IMPORTANCE (XGBoost Tuned)")
print("=" * 70)

fi = sorted(zip(feat_cols, best_xgb.feature_importances_), key=lambda x: -x[1])
for feat, imp in fi:
    bar = '█' * int(imp * 50)
    print(f"  {feat:20s} {imp:.4f}  {bar}")

CONFUSION MATRICES (Dev Set)
Logistic Regression        TN= 8786  FP=  0  FN= 33  TP=26325
SVM (LinearSVC)            TN= 8786  FP=  0  FN= 16  TP=26342
Random Forest              TN= 8786  FP=  0  FN= 12  TP=26346
XGBoost                    TN= 8786  FP=  0  FN= 11  TP=26347
Naive Bayes                TN= 8771  FP= 15  FN= 10  TP=26348
Ensemble (Soft Vote)       TN= 8786  FP=  0  FN= 11  TP=26347
XGBoost (Tuned)            TN= 8786  FP=  0  FN= 11  TP=26347

--- Test Set ---
Logistic Regression        TN= 8785  FP=  1  FN= 29  TP=26329
SVM (LinearSVC)            TN= 8785  FP=  1  FN= 20  TP=26338
Random Forest              TN= 8785  FP=  1  FN= 16  TP=26342
XGBoost                    TN= 8785  FP=  1  FN= 16  TP=26342
Naive Bayes                TN= 8777  FP=  9  FN= 18  TP=26340
Ensemble (Soft Vote)       TN= 8785  FP=  1  FN= 16  TP=26342
XGBoost (Tuned)            TN= 8785  FP=  1  FN= 16  TP=26342

FEATURE IMPORTANCE (XGBoost Tuned)
  correct_jaccard      0.9947  ██████████████████

In [10]:
print("=" * 70)
print("MODEL B2 — HINT SENTENCE SCORER")
print("=" * 70)

def build_hint_features(df, desc=""):
    rows = []
    labels = []
    for _, r in df.iterrows():
        article  = str(r['article'])
        question = str(r['question'])
        correct  = str(r[r['answer']])
        sents    = sentence_split(article)
        if not sents:
            continue
        key_ct = content_tokens(question) | content_tokens(correct)

        for si, sent in enumerate(sents):
            s_ct = content_tokens(sent)
            relevance = len(s_ct & key_ct) / max(len(key_ct), 1)
            rows.append({
                'position':        si / max(len(sents)-1, 1),
                'sent_word_count': len(tokenize(sent)),
                'q_jaccard':       jaccard(sent, question),
                'ans_jaccard':     jaccard(sent, correct),
                'q_overlap':       overlap_ratio(question, sent),
                'ans_overlap':     overlap_ratio(correct, sent),
                'keyword_overlap': relevance,
                'ans_in_sent':     int(clean_text(correct) in clean_text(sent)),
                'content_density': len(s_ct) / max(len(tokenize(sent)), 1),
            })
            labels.append(relevance)

    result_df = pd.DataFrame(rows)
    X = np.nan_to_num(result_df.values.astype(float), nan=0.0)
    y = np.array(labels)
    if desc:
        print(f"  {desc}: {X.shape[0]} sentences, {X.shape[1]} features")
    return X, y

train_hint_sample = train_df.sample(n=min(10000, len(train_df)), random_state=42)
Xh_train, yh_train = build_hint_features(train_hint_sample, "Train (sampled)")
Xh_dev,   yh_dev   = build_hint_features(dev_df, "Dev")

# Binary classifier
thr = np.percentile(yh_train, 80)
yh_train_bin = (yh_train >= thr).astype(int)
yh_dev_bin   = (yh_dev >= thr).astype(int)

print(f"\nBinary threshold: {thr:.4f}")
print(f"Train positives: {yh_train_bin.sum()} / {len(yh_train_bin)}")

hint_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
hint_lr.fit(Xh_train, yh_train_bin)
hp_dev = hint_lr.predict(Xh_dev)

print(f"\nHint Classifier (Logistic Regression):")
print(f"  Accuracy:  {accuracy_score(yh_dev_bin, hp_dev):.4f}")
print(f"  F1:        {f1_score(yh_dev_bin, hp_dev, average='macro'):.4f}")
print(f"  Precision: {precision_score(yh_dev_bin, hp_dev, average='macro'):.4f}")
print(f"  Recall:    {recall_score(yh_dev_bin, hp_dev, average='macro'):.4f}")
print(classification_report(yh_dev_bin, hp_dev, target_names=['Low Relevance', 'High Relevance']))
print(f"  Confusion Matrix: {confusion_matrix(yh_dev_bin, hp_dev).tolist()}")

# Regression scorer
hint_rf = RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
hint_rf.fit(Xh_train, yh_train)
yh_pred_dev = hint_rf.predict(Xh_dev)

r2   = r2_score(yh_dev, yh_pred_dev)
rmse = np.sqrt(mean_squared_error(yh_dev, yh_pred_dev))
mae  = mean_absolute_error(yh_dev, yh_pred_dev)
mse  = mean_squared_error(yh_dev, yh_pred_dev)

print(f"\nHint Regressor (Random Forest):")
print(f"  R2 Score:  {r2:.4f}")
print(f"  RMSE:      {rmse:.4f}")
print(f"  MAE:       {mae:.4f}")
print(f"  MSE:       {mse:.6f}")

MODEL B2 — HINT SENTENCE SCORER
  Train (sampled): 165296 sentences, 9 features
  Dev: 145102 sentences, 9 features

Binary threshold: 0.1429
Train positives: 37318 / 165296

Hint Classifier (Logistic Regression):
  Accuracy:  0.9953
  F1:        0.9934
  Precision: 0.9899
  Recall:    0.9970
                precision    recall  f1-score   support

 Low Relevance       1.00      0.99      1.00    112495
High Relevance       0.98      1.00      0.99     32607

      accuracy                           1.00    145102
     macro avg       0.99      1.00      0.99    145102
  weighted avg       1.00      1.00      1.00    145102

  Confusion Matrix: [[111821, 674], [2, 32605]]

Hint Regressor (Random Forest):
  R2 Score:  1.0000
  RMSE:      0.0001
  MAE:       0.0000
  MSE:       0.000000


In [11]:
print("=" * 70)
print("DISTRACTOR & HINT GENERATION PIPELINE")
print("=" * 70)

def generate_distractors(article, question, correct_answer, n_distractors=3):
    sents = sentence_split(article)
    correct_clean_t = clean_text(correct_answer)
    correct_wc = len(tokenize(correct_answer))
    correct_ct = content_tokens(correct_answer)
    q_ct = content_tokens(question)
    candidates = []

    for sent in sents:
        words = tokenize(sent)
        for n in range(max(1, correct_wc - 2), correct_wc + 3):
            for i in range(len(words) - n + 1):
                phrase = ' '.join(words[i:i+n])
                if phrase != correct_clean_t and len(phrase) > 2:
                    candidates.append(phrase)

    if correct_wc >= 5:
        for sent in sents:
            s_clean = clean_text(sent)
            if s_clean != correct_clean_t and 3 <= len(tokenize(s_clean)) <= correct_wc + 5:
                candidates.append(s_clean)

    art_tokens = tokenize(article)
    freq = Counter(art_tokens)
    content_freq = [(w, c) for w, c in freq.most_common(50) if w not in STOPWORDS]
    if correct_wc <= 3:
        for w, _ in content_freq[:20]:
            if w not in correct_ct:
                candidates.append(w)

    candidates = list(set(candidates))
    if not candidates:
        return ["Option A", "Option B", "Option C"][:n_distractors]

    scored = []
    for cand in candidates:
        cand_ct = content_tokens(cand)
        cand_wc = len(tokenize(cand))
        j = jaccard(cand, correct_answer)
        len_sim = max(1.0 - abs(cand_wc - correct_wc) / max(correct_wc, 1), 0.0)
        art_rel = overlap_ratio(cand, article)
        q_rel = len(cand_ct & q_ct) / max(len(cand_ct | q_ct), 1)
        exact_penalty = 1.0 if correct_clean_t not in cand and cand not in correct_clean_t else 0.1

        score = (0.15 * min(j, 0.5) * 2 + 0.25 * art_rel + 0.20 * q_rel
                 + 0.30 * len_sim + 0.10 * exact_penalty)
        if j > 0.8: score *= 0.1
        if cand == correct_clean_t: score = 0.0
        scored.append((cand, score))

    scored.sort(key=lambda x: -x[1])
    selected = []
    for cand, sc in scored:
        if len(selected) >= n_distractors: break
        if any(jaccard(cand, s) > 0.7 for s in selected): continue
        selected.append(cand)
    while len(selected) < n_distractors:
        selected.append(f"option {len(selected)+1}")
    return selected


def generate_hints(article, question, correct_answer, hint_model, n_hints=3):
    sents = sentence_split(article)
    if not sents:
        return ["Re-read the passage carefully."]
    key_ct = content_tokens(question) | content_tokens(correct_answer)

    scored_sents = []
    for si, sent in enumerate(sents):
        s_ct = content_tokens(sent)
        relevance = len(s_ct & key_ct) / max(len(key_ct), 1)
        feat = np.array([[
            si / max(len(sents)-1, 1), len(tokenize(sent)),
            jaccard(sent, question), jaccard(sent, correct_answer),
            overlap_ratio(question, sent), overlap_ratio(correct_answer, sent),
            relevance, int(clean_text(correct_answer) in clean_text(sent)),
            len(s_ct) / max(len(tokenize(sent)), 1),
        ]])
        score = hint_model.predict(feat)[0]
        scored_sents.append((sent, score))

    scored_sents.sort(key=lambda x: -x[1])
    key_list = list(key_ct)[:3]
    hints = []
    if len(scored_sents) >= 1:
        kw = key_list[0] if key_list else 'the key concept'
        hints.append(f"Hint 1 (General): The answer relates to '{kw}'. Look carefully in the passage.")
    if len(scored_sents) >= 2:
        hints.append(f"Hint 2 (Specific): Focus on this section: \"{scored_sents[1][0][:80]}...\"")
    if len(scored_sents) >= 3:
        hints.append(f"Hint 3 (Near-explicit): The answer is found near: \"{scored_sents[0][0][:120]}\"")
    return hints


# Demo on 5 dev examples
print("\n--- Demo ---\n")
for i in [0, 50, 200, 500, 1000]:
    if i >= len(dev_df): break
    row = dev_df.iloc[i]
    art, q, ans = str(row['article']), str(row['question']), row['answer']
    correct = str(row[ans])
    orig_dist = [str(row[o])[:60] for o in 'ABCD' if o != ans]

    gen_dist = generate_distractors(art, q, correct, 3)
    hints = generate_hints(art, q, correct, hint_rf, 3)

    print(f"Example {i}:")
    print(f"  Q: {q[:100]}")
    print(f"  Correct ({ans}): {correct[:80]}")
    print(f"  Original distractors: {orig_dist}")
    print(f"  Generated distractors: {[d[:60] for d in gen_dist]}")
    for h in hints:
        print(f"    {h}")
    print()

DISTRACTOR & HINT GENERATION PIPELINE

--- Demo ---

Example 0:
  Q: According to Dr Cerf's research, when a patient thinks about a famous film star,   _  .
  Correct (B): a particular neuron will become bright with excitement
  Original distractors: ['various neurons will identify the star as famous', 'the star will appear in his brain', "the star's face will light up"]
  Generated distractors: ['a particular neuron lit up when a volunteer', 'found that a particular neuron lit up when', 'neuron lit up when a volunteer thought about']
    Hint 1 (General): The answer relates to 'cerfs'. Look carefully in the passage.
    Hint 2 (Specific): Focus on this section: "If a database was built up identifying various neurons with concepts, objects an..."
    Hint 3 (Near-explicit): The answer is found near: "He found that a particular neuron lit up when a volunteer thought about Marilyn Monroe."

Example 50:
  Q: What would be the best title for the text?
  Correct (D): Background noise affect

In [12]:
print("=" * 70)
print("IMPROVED DISTRACTOR GENERATION — FREQUENCY-BASED SUBSTITUTION")
print("=" * 70)

def get_option_candidates_from_dataset(df, n=500):
    """Build a pool of real answer options from the dataset, grouped by word count."""
    pool = {}
    for _, r in df.sample(n=min(n, len(df)), random_state=42).iterrows():
        for opt in 'ABCD':
            text = str(r[opt]).strip()
            if text:
                wc = len(tokenize(text))
                if wc not in pool:
                    pool[wc] = []
                pool[wc].append(text)
    return pool

# Build option pool from training data
option_pool = get_option_candidates_from_dataset(train_df, n=5000)
print(f"Option pool built: {sum(len(v) for v in option_pool.values())} options across {len(option_pool)} length buckets")


def generate_distractors_v2(article, question, correct_answer, n_distractors=3):
    """
    Improved distractor generation with 3 strategies:
    1. Passage-based extraction (n-grams matching answer length)
    2. Frequency-based substitution (rubric requirement)
    3. Dataset pool sampling (fallback for natural-sounding options)
    """
    correct_clean_t = clean_text(correct_answer)
    correct_wc = len(tokenize(correct_answer))
    correct_ct = content_tokens(correct_answer)
    q_ct = content_tokens(question)
    sents = sentence_split(article)
    all_candidates = []

    # ── Strategy 1: Sentence-based extraction ──
    # Extract phrases from relevant sentences (not all sentences)
    q_keywords = content_tokens(question)
    relevant_sents = []
    for sent in sents:
        s_ct = content_tokens(sent)
        if len(s_ct & q_keywords) > 0:
            relevant_sents.append(sent)
    if not relevant_sents:
        relevant_sents = sents[:5]

    for sent in relevant_sents:
        words = tokenize(sent)
        for n in range(max(1, correct_wc), correct_wc + 2):
            for i in range(len(words) - n + 1):
                phrase = ' '.join(words[i:i+n])
                if phrase != correct_clean_t and len(phrase) > 2:
                    all_candidates.append(('extraction', phrase))

    # ── Strategy 2: Frequency-based substitution ──
    # Find content words in correct answer, substitute with frequent article words
    art_tokens_list = tokenize(article)
    freq = Counter(art_tokens_list)
    frequent_words = [w for w, c in freq.most_common(40) if w not in STOPWORDS]

    correct_words = list(correct_ct)
    for cw in correct_words:
        for fw in frequent_words:
            if fw != cw and fw not in correct_ct:
                # Substitute one content word
                new_phrase = clean_text(correct_answer).replace(cw, fw)
                if new_phrase != correct_clean_t:
                    all_candidates.append(('substitution', new_phrase))

    # For short answers (1-3 words), also use frequent words directly
    if correct_wc <= 3:
        for fw in frequent_words[:15]:
            if fw not in correct_ct:
                all_candidates.append(('substitution', fw))

    # ── Strategy 3: Dataset pool sampling ──
    # Get real options of similar length from the training data
    for wc_offset in [0, -1, 1, -2, 2]:
        target_wc = correct_wc + wc_offset
        if target_wc in option_pool:
            for opt_text in option_pool[target_wc][:30]:
                opt_clean = clean_text(opt_text)
                if opt_clean != correct_clean_t:
                    j = jaccard(opt_clean, correct_answer)
                    # Only include if somewhat relevant (shares some words with article/question)
                    art_rel = overlap_ratio(opt_clean, article)
                    if art_rel > 0.3 or j > 0.05:
                        all_candidates.append(('pool', opt_text.strip()))

    # Deduplicate
    seen = set()
    unique_candidates = []
    for source, cand in all_candidates:
        cand_clean = clean_text(cand)
        if cand_clean not in seen and cand_clean != correct_clean_t:
            seen.add(cand_clean)
            unique_candidates.append((source, cand))

    if not unique_candidates:
        return ["Option A", "Option B", "Option C"][:n_distractors]

    # ── Score all candidates ──
    scored = []
    for source, cand in unique_candidates:
        cand_ct = content_tokens(cand)
        cand_wc = len(tokenize(cand))

        j = jaccard(cand, correct_answer)
        len_sim = max(1.0 - abs(cand_wc - correct_wc) / max(correct_wc, 1), 0.0)
        art_rel = overlap_ratio(cand, article)
        q_rel = len(cand_ct & q_ct) / max(len(cand_ct | q_ct), 1)

        # Penalize exact substrings of correct answer
        exact_penalty = 0.1 if (correct_clean_t in clean_text(cand) or 
                                 clean_text(cand) in correct_clean_t) else 1.0
        # Penalize too-similar
        if j > 0.8:
            exact_penalty *= 0.1

        # Bonus for substitution strategy (grammatically similar to correct)
        source_bonus = 1.2 if source == 'substitution' else 1.0
        # Bonus for pool (natural-sounding)
        source_bonus = 1.1 if source == 'pool' else source_bonus

        score = source_bonus * (
            0.10 * min(j, 0.5) * 2
            + 0.20 * art_rel
            + 0.15 * q_rel
            + 0.40 * len_sim
            + 0.15 * exact_penalty
        )

        if clean_text(cand) == correct_clean_t:
            score = 0.0

        scored.append((cand, score, j, source))

    scored.sort(key=lambda x: -x[1])

    # Select with diversity
    selected = []
    selected_sources = []
    for cand, sc, j, source in scored:
        if len(selected) >= n_distractors:
            break
        if any(jaccard(cand, s) > 0.6 for s in selected):
            continue
        selected.append(cand)
        selected_sources.append(source)

    while len(selected) < n_distractors:
        selected.append(f"option {len(selected)+1}")
        selected_sources.append('fallback')

    return selected, selected_sources


# ── Demo with improved generation ──
print("\n--- Improved Distractor Generation Demo ---\n")
for i in [0, 50, 200, 500, 1000, 2000, 4000]:
    if i >= len(dev_df):
        break
    row = dev_df.iloc[i]
    art, q, ans = str(row['article']), str(row['question']), row['answer']
    correct = str(row[ans])
    orig_dist = [str(row[o])[:60] for o in 'ABCD' if o != ans]

    gen_dist, sources = generate_distractors_v2(art, q, correct, 3)

    print(f"Example {i}:")
    print(f"  Q: {q[:100]}")
    print(f"  Correct ({ans}): {correct[:80]}")
    print(f"  Original distractors: {orig_dist}")
    print(f"  Generated distractors:")
    for d, s in zip(gen_dist, sources):
        print(f"    [{s:12s}] {d[:70]}")
    print()


# ── Evaluate improved generation ──
print("=" * 70)
print("IMPROVED DISTRACTOR QUALITY EVALUATION")
print("=" * 70)

eval_n = min(500, len(dev_df))
not_correct = 0
total_d = 0
avg_art_ov2 = []
avg_len_r2 = []
avg_jac2 = []
source_counts = Counter()

for i in range(eval_n):
    row = dev_df.iloc[i]
    art, q = str(row['article']), str(row['question'])
    correct = str(row[row['answer']])
    correct_wc = len(tokenize(correct))

    gen, srcs = generate_distractors_v2(art, q, correct, 3)
    for d, s in zip(gen, srcs):
        total_d += 1
        source_counts[s] += 1
        if clean_text(d) != clean_text(correct):
            not_correct += 1
        avg_art_ov2.append(overlap_ratio(d, art))
        avg_len_r2.append(len(tokenize(d)) / max(correct_wc, 1))
        avg_jac2.append(jaccard(d, correct))

print(f"Samples: {eval_n} | Distractors: {total_d}")
print(f"Distractor ≠ correct (Accuracy): {not_correct/total_d:.4f}")
print(f"Avg article overlap:    {np.mean(avg_art_ov2):.4f}")
print(f"Avg length ratio:       {np.mean(avg_len_r2):.4f}")
print(f"Avg Jaccard w/ correct: {np.mean(avg_jac2):.4f}")
print(f"Source distribution:    {dict(source_counts)}")

IMPROVED DISTRACTOR GENERATION — FREQUENCY-BASED SUBSTITUTION
Option pool built: 19998 options across 27 length buckets

--- Improved Distractor Generation Demo ---

Example 0:
  Q: According to Dr Cerf's research, when a patient thinks about a famous film star,   _  .
  Correct (B): a particular neuron will become bright with excitement
  Original distractors: ['various neurons will identify the star as famous', 'the star will appear in his brain', "the star's face will light up"]
  Generated distractors:
    [substitution] a particular neuron will become dr with excitement
    [substitution] a particular neuron will according bright with excitement
    [substitution] a particular neuron will become bright with them

Example 50:
  Q: What would be the best title for the text?
  Correct (D): Background noise affects the taste of foods
  Original distractors: ['The best dinner environment', 'How to change plane foods', 'Listening to music while eating']
  Generated distractors:
    [sub

In [13]:
# ── Distractor Quality Evaluation ──
print("=" * 70)
print("DISTRACTOR QUALITY EVALUATION")
print("=" * 70)

eval_n = min(500, len(dev_df))
not_correct_count = 0
total_distractors = 0
avg_art_overlap = []
avg_len_ratio = []
avg_jaccard_correct = []

for i in range(eval_n):
    row = dev_df.iloc[i]
    art, q = str(row['article']), str(row['question'])
    correct = str(row[row['answer']])
    correct_wc = len(tokenize(correct))

    gen = generate_distractors(art, q, correct, 3)
    for d in gen:
        total_distractors += 1
        if clean_text(d) != clean_text(correct): not_correct_count += 1
        avg_art_overlap.append(overlap_ratio(d, art))
        avg_len_ratio.append(len(tokenize(d)) / max(correct_wc, 1))
        avg_jaccard_correct.append(jaccard(d, correct))

print(f"Samples: {eval_n} | Distractors: {total_distractors}")
print(f"Distractor ≠ correct (Accuracy): {not_correct_count/total_distractors:.4f}")
print(f"Avg article overlap:  {np.mean(avg_art_overlap):.4f}")
print(f"Avg length ratio:     {np.mean(avg_len_ratio):.4f}")
print(f"Avg Jaccard w/ correct: {np.mean(avg_jaccard_correct):.4f}")

# ── Save Models ──
print(f"\n{'=' * 70}")
print("SAVING MODELS")
print("=" * 70)

saves = {
    'distractor_lr.pkl': lr, 'distractor_svm.pkl': svm,
    'distractor_rf.pkl': rf, 'distractor_xgb.pkl': xgb_m,
    'distractor_xgb_tuned.pkl': best_xgb, 'distractor_nb.pkl': nb,
    'distractor_ensemble.pkl': ens, 'scaler.pkl': scaler,
    'hint_classifier_lr.pkl': hint_lr, 'hint_regressor_rf.pkl': hint_rf,
    'feature_columns.json': feat_cols,
}
for fname, obj in saves.items():
    path = os.path.join(SAVE_DIR, fname)
    if fname.endswith('.json'):
        with open(path, 'w') as f: json.dump(obj, f)
    else:
        joblib.dump(obj, path)
    print(f"  Saved: {path}")

# ── Final Summary ──
print(f"\n{'=' * 70}")
print("FINAL RESULTS — MODEL B1 (Distractor Ranker)")
print("=" * 70)
print(f"\n{'Model':<25} {'Dev Acc':>10} {'Dev F1':>10} {'Test Acc':>10} {'Test F1':>10}")
print("-" * 70)
for name, m in results_b1.items():
    print(f"{name:<25} {m['dev_accuracy']:>10.4f} {m['dev_f1']:>10.4f} "
          f"{m['test_accuracy']:>10.4f} {m['test_f1']:>10.4f}")

print(f"\nUnsupervised / Semi-Supervised:")
print(f"  K-Means Purity (dev):      {dev_purity:.4f}")
print(f"  K-Means Silhouette (dev):  {sil_dev:.4f}")
print(f"  Label Propagation Acc (dev):  {lp_acc_dev:.4f}")
print(f"  Label Propagation F1  (dev):  {lp_f1_dev:.4f}")
print(f"  Label Propagation Acc (test): {lp_acc_test:.4f}")
print(f"  Label Propagation F1  (test): {lp_f1_test:.4f}")

print(f"\nModel B2 — Hint Scorer (Dev Set):")
print(f"  Hint Classifier Accuracy:  {accuracy_score(yh_dev_bin, hp_dev):.4f}")
print(f"  Hint Classifier F1:        {f1_score(yh_dev_bin, hp_dev, average='macro'):.4f}")
print(f"  Hint Regressor R2:         {r2:.4f}")
print(f"  Hint Regressor RMSE:       {rmse:.4f}")
print(f"  Hint Regressor MAE:        {mae:.4f}")

elapsed = time.time() - start_time
print(f"\nTotal time: {elapsed:.1f}s")
print("=" * 70)
print("MODEL B TRAINING COMPLETE")
print("=" * 70)

DISTRACTOR QUALITY EVALUATION
Samples: 500 | Distractors: 1500
Distractor ≠ correct (Accuracy): 1.0000
Avg article overlap:  1.0000
Avg length ratio:     0.9984
Avg Jaccard w/ correct: 0.2059

SAVING MODELS
  Saved: /kaggle/working/models/model_b/distractor_lr.pkl
  Saved: /kaggle/working/models/model_b/distractor_svm.pkl
  Saved: /kaggle/working/models/model_b/distractor_rf.pkl
  Saved: /kaggle/working/models/model_b/distractor_xgb.pkl
  Saved: /kaggle/working/models/model_b/distractor_xgb_tuned.pkl
  Saved: /kaggle/working/models/model_b/distractor_nb.pkl
  Saved: /kaggle/working/models/model_b/distractor_ensemble.pkl
  Saved: /kaggle/working/models/model_b/scaler.pkl
  Saved: /kaggle/working/models/model_b/hint_classifier_lr.pkl
  Saved: /kaggle/working/models/model_b/hint_regressor_rf.pkl
  Saved: /kaggle/working/models/model_b/feature_columns.json

FINAL RESULTS — MODEL B1 (Distractor Ranker)

Model                        Dev Acc     Dev F1   Test Acc    Test F1
------------------